# NOCTIS — Sixteen imagined animal sounds

This notebook uses **AudioLDM**, a **pre-trained text-to-audio diffusion model** ([model card](https://huggingface.co/cvssp/audioldm-s-full-v2)) to generate **16 distinct sound effects**, with different prompts and different durations. The filename convention `call-01.wav` … `call-16.wav` matches `script.js` and the 16 fictional species. The sound model requires no additional training. It has a noncommercial ShareAlike model license; check license conditions for uses outside this class.

**In Colab:** choose **Runtime → Change runtime type → T4 GPU**, then run cells in order. The model weights download online the first time. Outputs have not been generated in this starter package.

In [ ]:
import torch
assert torch.cuda.is_available(), 'A CUDA GPU is required. Runtime → Change runtime type → T4 GPU.'
print('GPU:',torch.cuda.get_device_name(0))

## 1. Install packages

In [ ]:
%pip -q install "diffusers==0.35.1" "transformers==4.49.0" "accelerate==1.5.2" "huggingface_hub>=0.30,<1" "safetensors>=0.4" scipy soundfile

## 2. Prepare 16 acoustically different prompts

Descriptions mention **recognizable real-world sound sources** (bird, cat, frog, insect, etc.) because audio models respond more consistently to concrete acoustic language than to invented animal names. We then associate those sounds with our fictional species.

In [ ]:
species=[
 'Mossveil Drifter','Lantern-Throat Mothcat','Glasshorn Grazer','Velvet Bogling',
 'Thornback Mirrorkin','Raincall Strider','Saffron Treeglider','Moonroot Crawler',
 'Mistfin Anole','Cinder-Eyed Tapir','Hollowreed Staglet','Opalwing Skitter',
 'Bloomjaw Forager','Inkfeather Oracle','Sporetail Wanderer','Fernspark Nimbler'
]
sound_prompts=[
 'A peaceful low coo of a forest dove mixed with delicate leaf rustles, isolated animal call, no music',
 'A tiny wild cat purr and fluttering moth wings with crystalline chirps in the night forest, no music',
 'Two clear hollow whistles like an exotic deer call in thick morning fog, no music',
 'Small frog bubbling croaks and deep soft wetland gurgles in a quiet swamp, no music',
 'Sharp wooden insect clicks followed by a brief warning hiss from a jungle animal, no music',
 'An exotic tropical bird making rapid overlapping whistles under heavy rain, no music',
 'Small songbird rising chirps and rapidly fluttering wings high in tropical trees, no music',
 'A deep resonant rumble with distant burrowing scraping under the soil, no music',
 'An unusual tiny lizard squeaking with liquid plinks of water dripping from leaves, no music',
 'Soft breathing and rounded snorts from a forest tapir followed by a low purr, no music',
 'Airy bamboo flute tones produced by the wind, interspersed with a faraway bird chirp, no music',
 'A delicate rattling insect call with bright high pitched trills under the forest canopy, no music',
 'Tiny popping insect noises followed by a low amphibian croak in a flower garden, no music',
 'A distant raven caw with an eerie metallic echo in a dense jungle, no music',
 'Whispering tiny cracks and fungal rustling with a quiet rising bird trill, no music',
 'Fast squeaky tropical animal calls mixed with a gentle electric insect buzz, no music'
]
# Each sound has a different target duration, from 3.0 to 6.75 seconds.
durations=[round(3.0 + i*0.25,2) for i in range(16)]
for i,(name,seconds) in enumerate(zip(species,durations),1):print(f'{i:02d} {seconds:4.2f}s {name}')

## 3. Load pre-trained AudioLDM

This checkpoint is smaller than AudioLDM 2 and is a practical Colab starting point. You can switch to a different sound-diffusion model later and document the change.

In [ ]:
from diffusers import AudioLDMPipeline
MODEL_ID='cvssp/audioldm-s-full-v2'
pipe=AudioLDMPipeline.from_pretrained(MODEL_ID,torch_dtype=torch.float16)
pipe=pipe.to('cuda')
print('Model loaded:',MODEL_ID)

## 4. Generate 16 sounds

AudioLDM's requested output lengths can be quantized internally, so this code crops or zero-pads the actual generated waveform to **exactly** the assigned duration. Filenames match the website. Each prompt has its own deterministic seed. Review the files and regenerate any that sound unclear.

In [ ]:
import numpy as np
from pathlib import Path
from scipy.io import wavfile
import json

SOUNDS_DIR=Path('/content/sounds'); SOUNDS_DIR.mkdir(exist_ok=True)
SAMPLE_RATE=16000
NUM_INFERENCE_STEPS=35

for index,(name,prompt,seconds) in enumerate(zip(species,sound_prompts,durations)):
    generator=torch.Generator(device='cuda').manual_seed(7100+index)
    raw=pipe(prompt,negative_prompt='low quality, noise, distortion, music, speech',num_inference_steps=NUM_INFERENCE_STEPS,audio_length_in_s=seconds,guidance_scale=2.5,generator=generator).audios[0]
    raw=np.asarray(raw,dtype=np.float32).reshape(-1)
    target_samples=round(seconds*SAMPLE_RATE)
    if len(raw)<target_samples: raw=np.pad(raw,(0,target_samples-len(raw)))
    raw=raw[:target_samples]
    raw=np.nan_to_num(raw,nan=0.0,posinf=0.0,neginf=0.0)
    peak=max(float(np.max(np.abs(raw))),1e-6)
    output=np.int16(np.clip(raw/peak*0.88,-1,1)*32767)
    filename=SOUNDS_DIR/f'call-{index+1:02d}.wav'
    wavfile.write(filename,SAMPLE_RATE,output)
    print(f'Saved {filename.name} ({seconds:.2f}s) — {name}')

assert len(list(SOUNDS_DIR.glob('call-*.wav')))==16
(SOUNDS_DIR/'sound-prompts.json').write_text(json.dumps([{'name':n,'prompt':p,'duration_s':d} for n,p,d in zip(species,sound_prompts,durations)],indent=2))
print('Audio generation complete.')

## 5. Listen to a few results

In [ ]:
from IPython.display import Audio, display
for i in [1,5,9,13]:
    f=SOUNDS_DIR/f'call-{i:02d}.wav'
    print(species[i-1]);display(Audio(filename=str(f)))

## 6. Export (regular download **or** Google Drive)

Colab downloads sometimes get blocked. The Drive route is more reliable because you can open Drive separately and download the ZIP from there.

In [ ]:
from shutil import make_archive,copy2
zip_path=make_archive('/content/sounds-generated','zip',root_dir='/content',base_dir='sounds')
print('ZIP ready:',zip_path)
# OPTION A — Browser download:
# from google.colab import files
# files.download(zip_path)
# OPTION B — Copy to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# copy2(zip_path,'/content/drive/MyDrive/sounds-generated.zip')
# print('Saved in your My Drive.')

### Finish this part

1. Replace `postcards-from-my-jungle/sounds/call-01.wav` … `call-16.wav` with generated WAV files.
2. Save one screenshot of the inference log and note prompt revisions in your README.
3. Set `DEMO_MODE = false` in `script.js` only after all **50 images** and **16 sounds** are generated and swapped in.
4. Optionally publish `sound-prompts.json` in `sounds/` for reproducibility.